In [0]:
%pip install dbldatagen

In [0]:
try:
    import dbldatagen as dg
    print("dbldatagen is installed successfully!")
except ImportError:
    print("dbldatagen is not installed.")

In [0]:
import dbldatagen as dg
from pyspark.sql import functions as F

# Generate 1,000 synthetic customer records
data_generator = (
    dg.DataGenerator(spark, name="customers", rows=1000)
    .withIdOutput()
    .withColumn(
        "customer_id",
        "long",
        minValue=1000,
        maxValue=1999,
        random=False
    )
    .withColumn(
        "customer_name",
        "string",
        values=["Amit", "Neha", "Ravi", "Priya", "Rahul"],
        random=True
    )
    .withColumn(
        "city",
        "string",
        values=["Lucknow", "Delhi", "Mumbai", "Jaipur", "Kanpur"],
        random=True
    )
)

customers_df = data_generator.build()

display(customers_df.limit(10))

In [0]:
from pyspark.sql import functions as F

# Initial customer records
initial_events = (
    customers_df
    .withColumn("operation", F.lit("INSERT"))
    .withColumn("sequence_num", F.lit(1))
)

# Simulate updates for 100 customers
updates = (
    customers_df
    .filter(F.col("customer_id") < 1100)
    .withColumn(
        "city",
        F.when(F.col("city") == "Lucknow", "Kanpur")
         .otherwise("Lucknow")
    )
    .withColumn("operation", F.lit("UPDATE"))
    .withColumn("sequence_num", F.lit(2))
)

# Combine inserts and updates
customer_events = initial_events.unionByName(updates)

display(
    customer_events
    .filter(F.col("customer_id") < 1010)
    .orderBy("customer_id", "sequence_num")
)

In [0]:
(
    customer_events.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("default.customer_events_generated")
)

print("Generated customer events saved successfully.")

In [0]:
%sql
SELECT operation, COUNT(*) AS total
FROM default.customer_events_generated
GROUP BY operation;

In [0]:
%sql
SELECT
    operation,
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_id) AS unique_customers
FROM default.customer_events_generated
GROUP BY operation;